## Operational Flow 


In [1]:
import uuid

from sqlglot import parse_one, exp
from sqlglot.expressions import Expression

from owlready2 import get_ontology, sync_reasoner

### Step-1 : SQL string to AST object

In [2]:
class SqlglotOperator:
    """
    A dedicated class to encapsulate and operate on a sqlglot AST object.

    This operator automatically parses a SQL string, decorates the resulting
    AST with unique IDs for every node, and provides a suite of methods
    for easy and robust inspection and manipulation of the query structure.
    """

    def __init__(self, sql: str):
        """
        Initializes the operator by parsing the SQL and preparing the AST.
        """
        try:
            self.ast: Expression = parse_one(sql)
            self._decorate_ast_with_ids()
            self._map_ids_to_nodes()
        except Exception as e:
            print(f"Error initializing SqlglotOperator: {e}")
            self.ast = None
            self.id_to_node_map = {}

    def _decorate_ast_with_ids(self):
        """
        Private helper to traverse the AST and assign a unique ID to a 'meta'
        attribute on each node. This is fundamental for all ID-based operations.
        """
        if not self.ast:
            return
        for node in self.ast.walk():
            if not hasattr(node, 'meta'):
                node.meta = {}
            node.meta['id'] = str(uuid.uuid4())

    def _map_ids_to_nodes(self):
        """
        Private helper to create a dictionary mapping unique IDs to their
        corresponding nodes for efficient lookups.
        """
        if not self.ast:
            self.id_to_node_map = {}
            return
        self.id_to_node_map = {
            node.meta['id']: node for node in self.ast.walk() if hasattr(node, 'meta')
        }

    def to_sql(self, pretty: bool = True) -> str:
        """
        Generates the SQL string from the current state of the AST.
        """
        if not self.ast:
            return "-- Invalid AST"
        return self.ast.sql(pretty=pretty)

    def get_node_by_id(self, node_id: str) -> Expression | None:
        """
        Retrieves a single node from the AST by its unique ID.
        """
        return self.id_to_node_map.get(node_id)

    def get_nodes_by_type(self, node_type: type) -> list[Expression]:
        """
        Finds all nodes in the AST that are of a specific type.
        """
        if not self.ast:
            return []
        return self.ast.find_all(node_type)

    def remove_node_by_id(self, target_id: str):
        """
        Removes a single node from the AST based on its unique ID.
        The internal AST is updated with the result.
        """
        if not self.ast or not target_id:
            return

        def transformer(node):
            if hasattr(node, 'meta') and node.meta.get('id') == target_id:
                return None  # Returning None deletes the node
            return node

        # transform() returns a new AST object. We must update our instance.
        new_ast = self.ast.transform(transformer)
        self.ast = new_ast
        # After modification, the ID map needs to be rebuilt.
        self._map_ids_to_nodes()
        
    def add_node(self, parent_id: str, new_node: Expression, arg_name: str):
        """
        Adds a new node as a child of an existing node.
        """
        parent_node = self.get_node_by_id(parent_id)
        if not parent_node:
            print(f"[Error] Parent node with ID '{parent_id}' not found.")
            return

        # Decorate the new node and its children with IDs before adding
        self._decorate_ast_with_ids_on_subtree(new_node)

        # Handle list arguments (like 'expressions' in SELECT)
        if isinstance(getattr(parent_node, arg_name, None), list):
            parent_node.expressions.append(new_node)
        # Handle single arguments (like 'where' or 'limit')
        else:
            parent_node.set(arg_name, new_node)
        
        # Rebuild the map since we've added new nodes
        self._map_ids_to_nodes()

    def replace_node(self, target_id: str, new_node: Expression):
        """
        Replaces a target node with a new node.
        """
        if not self.ast or not target_id:
            return
            
        self._decorate_ast_with_ids_on_subtree(new_node)

        def transformer(node):
            if hasattr(node, 'meta') and node.meta.get('id') == target_id:
                return new_node
            return node
            
        self.ast = self.ast.transform(transformer)
        self._map_ids_to_nodes()

    def add_where_condition(self, condition_sql: str):
        """
        Adds a new condition to the main WHERE clause of the query from a string.
        """
        if not self.ast:
            print("[Error] AST is not initialized.")
            return

        # The .where() helper on the main query expression is the easiest way.
        # It handles both creating a new WHERE and AND-ing to an existing one.
        # copy=False ensures the modification happens in-place.
        self.ast.where(condition_sql, copy=False)

        # After modification, the AST structure has changed, so we need to
        # re-decorate the new parts and rebuild the ID map.
        self._decorate_ast_with_ids_on_subtree(self.ast)
        self._map_ids_to_nodes()

    def _decorate_ast_with_ids_on_subtree(self, ast_node: Expression):
        """Helper to add IDs to a new subtree before it's inserted."""
        for node in ast_node.walk():
            if not hasattr(node, 'meta'):
                node.meta = {}
            if 'id' not in node.meta:
                 node.meta['id'] = str(uuid.uuid4())

    def pretty_print(self):
        """Prints a visual, indented tree of the current AST with node IDs."""
        if not self.ast:
            print("-- Invalid AST")
            return
            
        for node in self.ast.walk():
            indent = "  " * node.depth
            short_id = node.meta.get('id', 'N/A')[:8]
            node_name = node.__class__.__name__
            
            # Add extra detail for common nodes
            if isinstance(node, exp.Identifier):
                node_name += f" ({node.this})"
                continue
            elif isinstance(node, exp.Column):
                 node_name += f" ({node.sql()})"

            print(f"{indent}├── {node_name}  [ID: {short_id}]")

### Step-2 : AST (`sqlgpt_parser`) to AST Tree conversion

In [3]:
class TreeNode:
    """ 
    TreeNode represents a node in the SQL AST tree. It can be a statement, clause, or expression.
    """
    def __init__(self, sqlglot_node, parent=None):
        self.remove = False
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = ASTTreeOperator.map_node_type(sqlglot_node, statement=True if parent is None else False)

        self.id = "n"+str(uuid.uuid4())[:6]
        sqlglot_node.meta['id'] = self.id

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            # self.reference_id = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable = parts[0].this
                self.refcol = parts[1].this
            elif len(parts) == 1:
                self.refcol = parts[0].this

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name
            if sqlglot_node.alias:
                self.refalias = sqlglot_node.alias
            # self.reference_id = None

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

        if self.name == "Literal":
            self.value = sqlglot_node.this
        
        if self.name == "Operator":
            if (type(sqlglot_node).__name__ in ["And", "Or"]):
                self.logics = True 
            else:
                self.logics = False    

    def add_child(self, child):
        self.children.append(child)

    def remove_child(self, child):
        if child in self.children:
            self.children.remove(child)
        else:
            raise ValueError(f"Child {child} not found in children of {self}")

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        if self.name == "Literal" and hasattr(self, "value"):
            suffix += f" [value={self.value}]"

        if hasattr(self, "reference_id"):
            suffix += f" [reference_id={self.reference_id}]"
        if hasattr(self, "remove"):
            suffix += f" [remove={self.remove}]"
        if hasattr(self, "refalias"):
            suffix += f" [refalias={self.refalias}]"
        if hasattr(self, "logics"):
            suffix += f" [logics={self.logics}]"
        return f"<({self.id}) {self.kind} | {suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)


class ASTTreeOperator:
    """
    Builds and operates on a simplified, custom tree structure derived
    from a sqlglot AST, making it easier to apply domain-specific logic.
    """
    # --- Type Mapper (integrated as a static method) ---
    CLAUSE_TYPES = {
        "From": "FromClause",
        "Group": "GroupByClause",
        "Into": "IntoClause",
        "Limit": "LimitClause",
        "Select": "SelectClause",
        "Set": "SetClause",
        "Update": "UpdateClause",
        "Values": "ValuesClause",
        "Where": "WhereClause",
        "Order": "OrderByClause",
        "Having": "HavingClause",
        "Join": "JoinClause"
    }

    STATEMENT_TYPES = {
        "Delete": "DeleteStatement",
        "Insert": "InsertStatement",
        "Update": "UpdateStatement",
        "Select": "SelectStatement"
    }
    EXPRESSION_CATEGORIES = {
        "Alias": "Alias", "Table": "TableRef", "Column": "ColumnRef", "Star": "Wildcard",
        "Identifier": "Identifier", "Sum": "Function", "Count": "Function", "Avg": "Function",
        "Max": "Function", "Min": "Function", "And": "Operator", "Or": "Operator", "EQ": "Operator",
        "GT": "Operator", "LT": "Operator", "GTE": "Operator", "LTE": "Operator", "Literal": "Literal"
    }

    @staticmethod
    def map_node_type(node: Expression, statement=False):
        """Maps a sqlglot AST node to a simplified (Kind, Name) tuple."""
        class_name = type(node).__name__
        if statement:
            if class_name in ASTTreeOperator.STATEMENT_TYPES:
                return "Statement", ASTTreeOperator.STATEMENT_TYPES[class_name]
            else:
                return "Statement", class_name
        else:
            if class_name in ASTTreeOperator.CLAUSE_TYPES:
                return "Clause", ASTTreeOperator.CLAUSE_TYPES[class_name]
            elif class_name in ASTTreeOperator.EXPRESSION_CATEGORIES:
                return "Expression", ASTTreeOperator.EXPRESSION_CATEGORIES[class_name]
            # Default fallback
            else:
                return "Expression", class_name

    def __init__(self, sql_operator: SqlglotOperator):
        """
        Initializes the tree operator using a pre-configured SqlglotOperator.
        """
        self.sql_op = sql_operator
        self.root: TreeNode | None = None
        self.id_to_node_map: dict[str, TreeNode] = {}
        if self.sql_op and self.sql_op.ast:
            self.build_tree(self.sql_op.ast)

    def create_expression_node(self, node_type, parent_sqlglot_node, parent_node, node_id, **kwargs):
        """
        Create a new ColumnRef node with the given reference column and table.
        """
        # first get node type related expression category dict reverse mapping
        if node_type in ASTTreeOperator.EXPRESSION_CATEGORIES.values():
            exp_class = next((k for k, v in ASTTreeOperator.EXPRESSION_CATEGORIES.items() if v == node_type), None)
        else:
            raise ValueError(f"Unknown node type: {node_type}")

        # create the sqlglot expression node
        new_exp = getattr(exp, exp_class)()
        # set node id 
        new_exp.meta['id'] = node_id
        # add node attributes from kwargs
        for key, value in kwargs.items():
            setattr(new_exp, key, value)

        # add exp to ast object 
        self.sql_op.add_node(parent_sqlglot_node.meta['id'], new_exp, parent_sqlglot_node.name)

        # create a new TreeNode for the custom tree
        new_tree_node = TreeNode(new_exp, parent=parent_node)

        parent_node.add_child(new_tree_node)
        return new_tree_node

    def build_tree(self, sqlglot_node, parent=None):
        # print(f"Building tree for node: {repr(sqlglot_node)} | Parent: {parent}")
        if parent is not None:
            clause_root = TreeNode(sqlglot_node, parent=parent)
            root_node = parent
            root_node.add_child(clause_root)
        else:
            root_node = TreeNode(sqlglot_node)
            self.root = root_node
            clause_root = TreeNode(sqlglot_node, parent=root_node)
            root_node.add_child(clause_root)
        

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]

            for expr in children:
                if not isinstance(expr, exp.Expression): # did not understand the logic of the check
                    continue

                c_kind, c_name = ASTTreeOperator.map_node_type(expr)

                if not found_first_clause and c_kind != "Clause":
                    sub = self._build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=root_node)
                    root_node.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier): # this is a weak check, so try to transfer this is recursion
                                continue  # Skip Identifier under Table
                            child_node = self._build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue  # Skip Identifier under Table
                                if isinstance(item, exp.Expression):
                                    child_node = self._build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
        if parent is None:
            return root_node

    def _build_recursive(self, sqlglot_node: Expression, parent_node: TreeNode = None) -> TreeNode | None:
        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "ColumnRef":
            return None

        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "TableRef":
            return None
        
        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "Alias":
            return None

        if isinstance(sqlglot_node, exp.TableAlias) and parent_node.name == "TableRef":
            return None

        if isinstance(sqlglot_node, exp.Paren):
            # return first child of the paren expression to avoid unnecessary nesting
            first_child = next(iter(sqlglot_node.args.values()))
            return self._build_recursive(first_child, parent_node)

        if isinstance(sqlglot_node, exp.Select):
            current = self.build_tree(sqlglot_node, parent_node)
        else:
            current = TreeNode(sqlglot_node, parent_node)
            for _, child in sqlglot_node.args.items():
                if isinstance(child, exp.Expression):
                    if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                        continue
                    child_node = self._build_recursive(child, current)
                    if child_node:
                        current.add_child(child_node)
                elif isinstance(child, list):
                    for item in child:
                        if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                            continue
                        if isinstance(item, exp.Expression):
                            child_node = self._build_recursive(item, current)
                            if child_node:
                                current.add_child(child_node)

        return current

    def walk(self):
        """Yields all nodes in the custom tree in pre-order traversal."""
        if not self.root:
            return
        
        nodes_to_visit = [self.root]
        while nodes_to_visit:
            current = nodes_to_visit.pop(0)
            yield current
            nodes_to_visit.extend(current.children)

    def get_node_by_id(self, node_id: str) -> TreeNode | None:
        """Gets a TreeNode from the custom tree by its ID."""
        return self.id_to_node_map.get(node_id)

    def get_parent(self, node_id: str) -> TreeNode | None:
        """Gets the immediate parent of a node."""
        node = self.get_node_by_id(node_id)
        return node.parent if node else None

    def get_parent_clause(self, node_id: str) -> TreeNode | None:
        """Finds the first ancestor of a node that is a 'Clause'."""
        current_node = self.get_node_by_id(node_id)
        while current_node:
            if current_node.kind == "Clause":
                return current_node
            current_node = current_node.parent
        return None

    def get_parent_statement(self, node_id: str) -> TreeNode | None:
        """Finds the first ancestor of a node that is a 'Statement'."""
        current_node = self.get_node_by_id(node_id)
        while current_node:
            if current_node.kind == "Statement":
                return current_node
            current_node = current_node.parent
        return None

    def get_tables_in_statement(self, stmt_node: TreeNode) -> list[TreeNode]:
        """Finds all TableRef TreeNodes within a given Statement TreeNode."""
        tables = []
        if stmt_node.kind != "Statement":
            return []
        
        from_clause = next((c for c in stmt_node.children if c.name == 'From'), None)
        if from_clause:
            for node in from_clause.children:
                if node.name == "TableRef":
                    tables.append(node)
        return tables

    def pretty_print(self):
        """Prints a visual, indented tree of the custom AST."""
        if not self.root:
            print("-- Tree is not built --")
            return
        
        def print_recursive(node, prefix="", is_last=True):
            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")
            
            for i, child in enumerate(node.children):
                new_prefix = prefix + ("    " if is_last else "│   ")
                print_recursive(child, new_prefix, is_last=(i == len(child.parent.children) - 1))
        
        print_recursive(self.root)


### Step-3 : AST Tree to Ontology Instance 

with Ontology reasoning

In [54]:
class OntologyOperator:
    """Orchestrates the resolution and instantiation of an ontology from an AST tree."""
    def __init__(self, tree_operator: ASTTreeOperator, onto_path: str):
        self.tree_op = tree_operator
        self.onto = get_ontology(onto_path).load()
        self.class_lookup = {cls.name: cls for cls in self.onto.classes()}
        self.created_instances = []
        self.table_ref_instances = []
        self.column_ref_instances = []

    def get_table_ref_instances(self):
        """
        Returns the list of instantiated TableRef objects.
        """
        return {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in self.table_ref_instances }
    
    def get_column_ref_instances(self):
        """
        Returns the list of instantiated ColumnRef objects.
        """
        
        return {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in self.column_ref_instances }

    def get_individuals_by_class(self, class_name: str) -> list:
        """
        Gets all individuals that are instances of a given class name.

        This correctly handles that an individual can belong to multiple classes.

        Args:
            class_name (str): The name of the class to search for (e.g., "TableRef").

        Returns:
            A list of all matching owlready2 individuals.
        """
        # Find the actual class object from the lookup map created in __init__
        target_class = self.class_lookup.get(class_name)

        if not target_class:
            print(f"[Warning] Class '{class_name}' not found in the ontology.")
            return []

        # The onto.search() method efficiently finds all individuals of the given type.
        return self.onto.search(type=target_class)
    
    def resolve_wildcard(self):
        """
        Finds and expands all wildcards in the query. This modifies the
        underlying AST and rebuilds the custom tree. Call before instantiation.
        """
        wildcard_nodes = [node for node in self.tree_op.walk() if node.name == "Wildcard"]
        if not wildcard_nodes:
            return

        print(f"--> Found {len(wildcard_nodes)} wildcard(s). Expanding...")
        for wc_node in wildcard_nodes:
            parent_node = self.tree_op.get_parent(wc_node.id)
            parent_stmt_treenode = self.tree_op.get_parent_statement(wc_node.id)
            if not parent_stmt_treenode: return

            tables_in_stmt = self.tree_op.get_tables_in_statement(parent_stmt_treenode)
            if len(tables_in_stmt) != 1:
                print(f"[Warning] Wildcard expansion only supported for single-table queries. Skipping.")
                return

            wildcard_sqlglot_node = self.tree_op.sql_op.get_node_by_id(wc_node.id)
            parent_select_sqlglot_node = wildcard_sqlglot_node.find_ancestor(exp.Select)
            
            table_name = tables_in_stmt[0].reftable
            table_ind = self.onto.search_one(TableName=table_name)
            if not table_ind or not hasattr(table_ind, "hasColumn"):
                print(f"[Warning] Table '{table_name}' not in ontology or has no columns. Cannot expand.")
                return
            
            # go through all columns of the table and create ColumnRef nodes
            for col in table_ind.hasColumn:
                col_name = col.ColumnName[0]
                new_node_id = "n" + str(uuid.uuid4())[:6]
                new_col_node = self.tree_op.create_expression_node(
                    node_type="ColumnRef",
                    parent_sqlglot_node=parent_select_sqlglot_node,
                    parent_node=parent_node,
                    node_id=new_node_id,
                    refcol=col_name,
                    reftable=table_name
                )

            # Remove the wildcard node from the parent statement
            self.tree_op.remove_node_by_id(wc_node.id)

    def resolve_references(self):
        """
        Starts the recursive, scope-aware process to resolve all table and
        column references in the custom tree against the ontology.
        """
        print("--> Resolving references against ontology (Recursive, Scope-Aware)...")
        # Prepare the global lookups from the ontology once.
        table_entities = {t.TableName[0]: t.name for t in self.get_individuals_by_class("Table")}
        column_lookup = {(tbl.TableName[0], c.ColumnName[0]): c.name for c in self.get_individuals_by_class("Column") for tbl in c.columnOfTable}
        
        # Start the recursion from the root of the tree with an empty initial context.
        if self.tree_op.root:
            self._resolve_recursive(self.tree_op.root, {}, table_entities, column_lookup)

    def _resolve_recursive(self, current_node: 'TreeNode', parent_context: dict, table_entities: dict, column_lookup: dict) -> dict:
        """
        Recursively resolves references using a two-pass, post-order traversal.
        It passes parent context down and returns defined aliases up.

        Args:
            current_node: The TreeNode currently being processed.
            parent_context: The context (tables and aliases) from outer scopes.
            table_entities: Global lookup for table names to ontology IDs.
            column_lookup: Global lookup for (table, column) to ontology IDs.

        Returns:
            A dictionary of aliases defined by the current node's scope.
        """
        # 1. ESTABLISH CONTEXT TO PASS DOWN
        # Get tables/aliases from the current node's own FROM clause, if it's a statement/subquery.
        local_context = self._get_statement_context(current_node)
        # The context for children is the parent's context merged with our local one.
        child_context = self._merge_contexts(parent_context, local_context)

        # 2. PASS 1: RECURSE ON SUBQUERIES (POST-ORDER)
        # Resolve all child subqueries first to learn what aliases they define.
        aliases_from_subqueries = {}
        for child in current_node.children:
            subquery_node, alias_name = None, None
            if child.name == "Alias" and child.children and child.children[0].name == "Subquery":
                subquery_node, alias_name = child.children[0], child.sqlglot_node.alias
            elif child.name == "Subquery":
                subquery_node = child

            if subquery_node:
                defined_aliases = self._resolve_recursive(subquery_node, child_context, table_entities, column_lookup)
                if alias_name:
                    aliases_from_subqueries[alias_name] = defined_aliases
                else:
                    aliases_from_subqueries.update(defined_aliases)

        # 3. FINALIZE FULL CONTEXT FOR THIS LEVEL
        # This includes tables from all parent/local scopes and aliases from child subqueries.
        full_context_for_resolution = {
            "tables": child_context,
            "subquery_aliases": aliases_from_subqueries
        }

        # 4. PASS 2: RESOLVE LOCAL NODES
        # Walk this node's children again, but this time resolve the simple nodes
        # and explicitly DO NOT enter subqueries.
        nodes_to_process = list(current_node.children)
        while nodes_to_process:
            node = nodes_to_process.pop(0)
            if node.name == "Subquery": continue # Already processed in Pass 1

            if node.name == "TableRef" and hasattr(node, "reftable"):
                node.reference_id = table_entities.get(node.reftable)
            elif node.name == "ColumnRef" and hasattr(node, "refcol"):
                self._resolve_single_column(node, full_context_for_resolution, column_lookup)
            
            nodes_to_process.extend(node.children)

        # 5. RETURN DEFINED ALIASES TO PARENT
        # Find aliases defined in the current node's SELECT list to return them.
        aliases_defined_at_this_level = {}
        if current_node.kind == "Statement" or current_node.name == "Subquery":
            select_clause = next((c for c in current_node.children if c.name == 'Select'), None)
            if select_clause:
                for expr_node in select_clause.children:
                    if expr_node.name == "Alias":
                        alias_name = expr_node.sqlglot_node.alias
                        if expr_node.children:
                            aliased_node = expr_node.children[0]
                            aliases_defined_at_this_level[alias_name] = getattr(aliased_node, 'reference_id', aliased_node.id)
        
        return aliases_defined_at_this_level

    def _resolve_single_column(self, node: 'TreeNode', context: dict, column_lookup: dict):
        """Resolves a single ColumnRef node using the fully built context."""
        table_context = context.get("tables", {})
        subquery_aliases = context.get("subquery_aliases", {})
        alias_map = table_context.get("alias_map", {})
        
        if hasattr(node, "reftable") and node.reftable:
            table_alias = node.reftable
            # Case 1: Is the alias a table from a FROM clause? (e.g., c.name)
            if table_alias in alias_map:
                tbl_key = alias_map.get(table_alias)
                node.reference_id = column_lookup.get((tbl_key, node.refcol))
            # Case 2: Is the alias a subquery? (e.g., sub.total_orders)
            elif table_alias in subquery_aliases:
                subquery_defined_aliases = subquery_aliases.get(table_alias, {})
                node.reference_id = subquery_defined_aliases.get(node.refcol)
        # Case 3: No table alias provided, resolve if only one table is in scope.
        elif len(table_context.get("tables_in_scope", [])) == 1:
            tbl_key = table_context["tables_in_scope"][0]
            node.reference_id = column_lookup.get((tbl_key, node.refcol))

    def _merge_contexts(self, parent_context: dict, local_context: dict) -> dict:
        """Helper to merge parent and local contexts, with local taking precedence."""
        return {
            "alias_map": {**parent_context.get("alias_map", {}), **local_context.get("alias_map", {})},
            "tables_in_scope": parent_context.get("tables_in_scope", []) + local_context.get("tables_in_scope", [])
        }
    
    def _get_statement_context(self, stmt_node: 'TreeNode') -> dict:
        """
        Helper to get the local table context (aliases and tables) for a
        single Statement or Subquery TreeNode.

        Args:
            stmt_node: The Statement or Subquery TreeNode to analyze.

        Returns:
            A dictionary containing the local context for this scope.
        """
        alias_map = {}
        tables_in_scope = []

        # A scope is only defined by a Statement or a Subquery that has children.
        if (stmt_node.kind != "Statement" and stmt_node.name != "Subquery") or not stmt_node.children:
            return {"alias_map": {}, "tables_in_scope": []}

        # Find the FROM clause directly under the statement node
        from_clause = next((c for c in stmt_node.children if c.name == 'From'), None)
        
        if from_clause:
            # Walk through the direct children of the FROM clause to find all table references and their aliases
            for t_node in from_clause.children:
                # This handles cases like `... FROM customers AS c`
                if t_node.name == "TableRef" and hasattr(t_node, "reftable"):
                    table_name = t_node.reftable
                    tables_in_scope.append(table_name)
                    
                    # The table's own name is a valid way to refer to it
                    alias_map[table_name] = table_name
                    
                    # If an alias exists (e.g., 'AS c'), map it to the real table name
                    if hasattr(t_node, "refAlias"):
                        alias_map[t_node.refAlias] = table_name
                
                # This handles aliased subqueries in the FROM clause, e.g., `... FROM (SELECT...) AS sub`
                elif t_node.name == "Alias" and t_node.children and t_node.children[0].name == "Subquery":
                    # The alias (e.g., 'sub') refers to the subquery. We map the alias name
                    # to itself, as it represents a new data source, not a direct table.
                    # The columns it provides will be resolved by the recursive return.
                    alias_name = t_node.sqlglot_node.alias
                    alias_map[alias_name] = alias_name # Map the alias to itself for context
                        
        return {"alias_map": alias_map, "tables_in_scope": tables_in_scope}

    def instantiate_ontology(self, agent_id: str):
        """Walks the tree and creates ontology individuals."""
        print("--> Instantiating ontology individuals...")
        self.created_instances = []
        self.table_ref_instances = []
        self.column_ref_instances = []
        with self.onto:
            self.resolve_wildcard()
            self.resolve_references()
            
            self._instantiate_recursive(self.tree_op.root, agent_id)
        print(f"    - Instantiated {len(self.table_ref_instances)} TableRefs and {len(self.column_ref_instances)} ColumnRefs.")

    def _instantiate_recursive(self, node: TreeNode, agent_id: str, parent_instance=None):
        node_class = self.class_lookup.get(node.name)
        if not node_class: return
        
        inst = node_class(node.id)
        self.created_instances.append(inst)
        if parent_instance: inst.immediateParentNode.append(parent_instance)

        if node.kind == "Statement":
            agent_ref = self.onto.search_one(iri=f"*{agent_id}")
            if agent_ref: inst.executedBy.append(agent_ref)
        
        if hasattr(node, "reference_id") and node.reference_id:
            ref_obj = self.onto.search_one(iri=f"*{node.reference_id}")
            if ref_obj:
                if node.name == "TableRef":
                    inst.referencesToTable.append(ref_obj)
                    self.table_ref_instances.append(inst)
                elif node.name == "ColumnRef":
                    inst.referencesToColumn.append(ref_obj)
                    self.column_ref_instances.append(inst)

        for child in node.children:
            self._instantiate_recursive(child, agent_id, parent_instance=inst)

    def reason_and_save(self, output_path: str):
        """Runs the reasoner and saves the final ontology."""
        print("--> Running reasoner and saving ontology...")
        with self.onto:
            sync_reasoner(infer_property_values=True)
        self.onto.save(file=output_path, format="rdfxml")
        print(f"    - Ontology saved to {output_path}")


In [5]:
class ASTPruner:
    """
    Prunes a sqlglot AST based on the state of an associated ontology.
    """
    def __init__(self, onto_op: OntologyOperator):
        self.onto_op = onto_op
        self.sql_op = onto_op.tree_op.sql_op
        self.tree_op = onto_op.tree_op
        self.violated_ids = set()

    def prune(self):
        """
        Orchestrates the full, multi-stage pruning process.
        The underlying AST is modified in place.
        """
        print("--- Starting AST Pruning Process ---")
        self._prune_from_table_status()
        self._prune_from_column_status()
        
        # After all modifications, rebuild the custom tree to reflect changes
        # print("--> Pruning complete. Rebuilding custom tree...")
        # self.tree_op.build_tree()
        # print("--- Pruning Process Finished ---")

    def _prune_from_table_status(self):
        """
        Stage 1: Handles removals and modifications based on TableRef statuses.
        """
        print("--> Stage 1: Pruning based on table statuses...")
        statements_to_remove_ids = set()

        for inst in self.onto_op.table_ref_instances:
            status_list = getattr(inst, "hasStatus", [])
            if not status_list: continue
            status = status_list[0]

            # Condition A.1: Violated Table -> Remove Statement
            if status == "Violated":
                parent_stmt = self.tree_op.get_parent_statement(inst.name)
                if parent_stmt:
                    print(f"    - TableRef '{inst.name}' is Violated. Marking statement '{parent_stmt.id[:8]}' for removal.")
                    statements_to_remove_ids.add(parent_stmt.id)

            # Condition A.2: RowTag Table -> Add WHERE condition
            elif status == "RowTag":
                policy_list = getattr(inst, "relatedPolicy", [])
                if not policy_list: continue
                
                condition_list = getattr(policy_list[0], "hasActionCondition", [])
                if not condition_list: continue
                
                condition_str = condition_list[0]
                parent_stmt = self.tree_op.get_parent_statement(inst.name)
                if parent_stmt and parent_stmt.name == "Select":
                    print(f"    - TableRef '{inst.name}' has RowTag. Adding condition to statement '{parent_stmt.id[:8]}': {condition_str}")
                    stmt_sqlglot_node = self.sql_op.get_node_by_id(parent_stmt.id)
                    if stmt_sqlglot_node:
                        stmt_sqlglot_node.where(condition_str, copy=False)

        if statements_to_remove_ids:
            # Use a transformer to remove all marked statements at once
            def stmt_remover(node):
                if hasattr(node, 'meta') and node.meta.get('id') in statements_to_remove_ids:
                    return None
                return node
            self.sql_op.ast = self.sql_op.ast.transform(stmt_remover)

    def _prune_from_column_status(self):
        """
        Stage 2: Handles cascading removals based on ColumnRef 'Violated' status.
        """
        print("--> Stage 2: Pruning based on column statuses...")
        # Step 1: Collect initial violated IDs and identify tainted aliases
        self.violated_ids = {inst.name for inst in self.onto_op.column_ref_instances if getattr(inst, "hasStatus", [""])[0] == "Violated"}
        if not self.violated_ids:
            print("    - No violated columns found.")
            return
        
        print(f"    - Initial violated column IDs: {self.violated_ids}")
        self._propagate_violations_through_aliases()

        # Step 2: Apply the main pruning transformer with all collected IDs
        self.sql_op.ast = self.sql_op.ast.transform(self._pruning_transformer)

    def _propagate_violations_through_aliases(self):
        """Finds columns that are aliased and marks their usages as violated."""
        # This is a simplified propagation. A full implementation for complex
        # nested queries would require more sophisticated scope analysis.
        tainted_aliases = set()
        for alias_node in self.sql_op.ast.find_all(exp.Alias):
            source_col_id = alias_node.this.meta.get('id')
            if source_col_id in self.violated_ids:
                tainted_aliases.add(alias_node.alias)
        
        if tainted_aliases:
            print(f"    - Propagating violations from tainted aliases: {tainted_aliases}")
            for col_node in self.sql_op.ast.find_all(exp.Column):
                if col_node.this.name in tainted_aliases:
                    self.violated_ids.add(col_node.meta.get('id'))

    def _pruning_transformer(self, node):
        """The core transformation logic for cascading removals."""
        # Base Case: If the node ID is in our final violated set, remove it.
        if hasattr(node, 'meta') and node.meta.get('id') in self.violated_ids:
            print(f"    - Removing node {node.sql()} [ID: {node.meta.get('id')[:8]}]")
            return None

        # Recursive Cases for cascading removals
        if isinstance(node, exp.Func) and not node.this:
            print(f"    - Cascading removal of Function '{node.sql()}'")
            return None
        if isinstance(node, exp.Binary) and (node.left is None or node.right is None):
            print(f"    - Cascading removal of Operator '{node.sql()}'")
            return None
        if isinstance(node, exp.Join) and node.on is None:
            print(f"    - Cascading removal of JOIN '{node.sql()}'")
            return None
        if isinstance(node, exp.Select) and not node.expressions:
            print(f"    - All columns removed from a SELECT clause.")

        return node

---
## Run

In [43]:
# query = "SELECT bank, SUM(amount), type AS total FROM Transaction WHERE (account_id = 123) and (amount > 100) GROUP BY account_id"
# query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

query = """
SELECT
    c.duration,
    c.date,
    (
        SELECT COUNT(*)
        FROM Order o
        WHERE o.account_id = c.account_id
    ) AS total_orders,
    (
        SELECT AVG(o2.amount)
        FROM Order o2
        WHERE o2.account_id = c.account_id
          AND o2.amount > 100
    ) AS avg_recent_order_amount
FROM Loan c
WHERE c.status = 'active';
"""

# query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

In [44]:
sql_op = SqlglotOperator(query)
# view the sqlglot AST
print("SQLGlot AST:")
# sql_op.pretty_print()

SQLGlot AST:


In [45]:
sql_op.ast

Select(
  expressions=[
    Column(
      this=Identifier(this=duration, quoted=False),
      table=Identifier(this=c, quoted=False)),
    Column(
      this=Identifier(this=date, quoted=False),
      table=Identifier(this=c, quoted=False)),
    Alias(
      this=Subquery(
        this=Select(
          expressions=[
            Count(
              this=Star(),
              big_int=True)],
          from=From(
            this=Table(
              this=Identifier(this=Order, quoted=False),
              alias=TableAlias(
                this=Identifier(this=o, quoted=False)))),
          where=Where(
            this=EQ(
              this=Column(
                this=Identifier(this=account_id, quoted=False),
                table=Identifier(this=o, quoted=False)),
              expression=Column(
                this=Identifier(this=account_id, quoted=False),
                table=Identifier(this=c, quoted=False)))))),
      alias=Identifier(this=total_orders, quoted=False)),
    A

In [46]:
tree_op = ASTTreeOperator(sql_op)
print("\nCustom AST Tree:")
tree_op.pretty_print()


Custom AST Tree:
└── <(n0ef455) Statement |  (SelectStatement) [remove=False]>
    ├── <(ndbeeca) Clause |  (SelectClause) [remove=False]>
    │   ├── <(n88743f) Expression |  (ColumnRef) [reftable=c, refcol=duration] [remove=False]>
    │   ├── <(nc27722) Expression |  (ColumnRef) [reftable=c, refcol=date] [remove=False]>
    │   ├── <(ne110c2) Expression |  (Alias) [name=total_orders] [remove=False]>
    │   │   └── <(n1dc8ac) Expression |  (Subquery) [remove=False]>
    │   │       ├── <(n3c6844) Clause |  (SelectClause) [remove=False]>
    │   │       │   └── <(n52350f) Expression |  (Function) [remove=False]>
    │   │       │       └── <(n5a35bc) Expression |  (Wildcard) [remove=False]>
    │   │       ├── <(n276226) Clause |  (FromClause) [remove=False]>
    │   │       │   └── <(n6d923b) Expression |  (TableRef) [reftable=Order] [remove=False] [refalias=o]>
    │   │       └── <(na295c5) Clause |  (WhereClause) [remove=False]>
    │   │           └── <(n38493c) Expression |  (

In [55]:
onto_path = "../ontology_file/aput.rdf"
output_path = "../ontology_file/resolved_query_instances.rdf"
agent_id = "a0012"

# 1. Create an instance of the class
instantiator = OntologyOperator(tree_op, onto_path)

# 2. Instantiate the ontology
# 2.a resolve wildcards in the query
# 2.b resolve references in the tree
# 2.c instantiate the ontology individuals
instantiator.instantiate_ontology(agent_id)
tree_op.pretty_print()
# 3. Run the reasoner and save the ontology
# instantiator.reason_and_save(output_path)

--> Instantiating ontology individuals...
--> Found 1 wildcard(s). Expanding...
--> Resolving references against ontology (Recursive, Scope-Aware)...
    - Instantiated 1 TableRefs and 0 ColumnRefs.
└── <(n0ef455) Statement |  (SelectStatement) [remove=False]>
    ├── <(ndbeeca) Clause |  (SelectClause) [remove=False]>
    │   ├── <(n88743f) Expression |  (ColumnRef) [reftable=c, refcol=duration] [remove=False]>
    │   ├── <(nc27722) Expression |  (ColumnRef) [reftable=c, refcol=date] [remove=False]>
    │   ├── <(ne110c2) Expression |  (Alias) [name=total_orders] [remove=False]>
    │   │   └── <(n1dc8ac) Expression |  (Subquery) [remove=False]>
    │   │       ├── <(n3c6844) Clause |  (SelectClause) [remove=False]>
    │   │       │   └── <(n52350f) Expression |  (Function) [remove=False]>
    │   │       │       └── <(n5a35bc) Expression |  (Wildcard) [remove=False]>
    │   │       ├── <(n276226) Clause |  (FromClause) [remove=False]>
    │   │       │   └── <(n6d923b) Expression 

In [11]:
pruner = ASTPruner(instantiator)

# 4. Prune the AST based on the ontology state
pruner.prune()

sql_op.to_sql()

--- Starting AST Pruning Process ---
--> Stage 1: Pruning based on table statuses...
--> Stage 2: Pruning based on column statuses...
    - No violated columns found.


"SELECT\n  c.customer_id,\n  c.name,\n  (\n    SELECT\n      COUNT(*)\n    FROM orders AS o\n    WHERE\n      o.customer_id = c.customer_id\n  ) AS total_orders,\n  (\n    SELECT\n      AVG(o2.total_amount)\n    FROM orders AS o2\n    WHERE\n      o2.customer_id = c.customer_id\n      AND o2.order_date > CURRENT_DATE - INTERVAL '30' DAYS\n  ) AS avg_recent_order_amount\nFROM customers AS c\nWHERE\n  c.status = 'active'"